# Save tagged-unit figures

Independent notebook that loads one opto-tagging session, selects **robustly tagged units**,
and saves **one figure per unit**. Every condition for a unit is drawn in the **same figure**
(3-column grid), with the unit's mean waveform as an inset in the top-right corner. Figures are
saved in two alignments:

- **Pulse-aligned** (step 5) — aligned to each pulse; each condition panel is annotated with the
  pooled `reliability`, `median_latency`, `jitter` and `effect_ratio` for that unit/condition.
- **Train-aligned** (step 6) — aligned to each laser train, shading every pulse so the whole-train
  response is visible.

Tagging is split into two independent steps so metrics are computed once and selection
thresholds can be explored freely:

1. **Compute & save** (`OpticalTagging.compute_tagging_metrics`) — calculates the raw metrics
   for every unit × condition and, with `per_pulse=True`, **separately for each pulse position
   within the train** (`pulse_index = -1` is the pooled aggregate; `0, 1, 2, …` are individual
   pulses). Each row carries a `session` column (the NWB `session_id`) and the table is saved to
   a per-session CSV.
2. **Select** (`select_tagged_units`, a **standalone module-level function**) — applies the
   threshold criteria and adds the `tagged` flag. Re-run this alone to try new cut-offs. This
   notebook handles a **single session**; the saved CSVs can still be pooled across sessions and
   selected together elsewhere, since the function needs no `OpticalTagging` instance.

The layered criteria (on top of a paired t-test) avoid over-calling tagged units when trial
counts are large:

- **Short-latency window** (0–6 ms) for direct activation
- **SALT test** (Kvitsiani et al. 2013) on first-spike-latency distributions
- **Effect size**: `stim_FR ≥ k × baseline_FR`
- **Reliability**: fraction of pulses evoking a spike in the window
- **Latency / jitter**: small median first-spike latency and low SD
- **Multiple-comparison correction** (Benjamini–Hochberg FDR) on both the t-test and SALT
  p-values, applied **within each `pulse_index` family**

Figures are saved as **PNG by default** and rendered **in parallel** across worker processes.

In [7]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Analysis modules loaded from: /root/capsule/src/aind_dft_ephys_analysis


In [8]:
# =============================================================================
# 2. LOAD SESSION
# =============================================================================
import glob
from optical_tagging import OpticalTagging

session_date = '863973_2026-08-25_14-18-29'
behavior_json_file = f'/root/capsule/data/ecephys_{session_date}/behavior/{session_date}.json'

# The sorted folder has an extra timestamp not contained in session_date,
# so locate it by glob (one match expected per session).
ephys_nwb_file = glob.glob(
    f'/root/capsule/data/ecephys_{session_date}_sorted_*/nwb/ecephys_{session_date}_experiment1_recording1.nwb'
)[0]

OptoTag = OpticalTagging(
    behavior_json_file=behavior_json_file,
    ephys_nwb_file=ephys_nwb_file,
)

# Where the computed (uncorrected) tagging metrics will be saved.
metrics_csv_path = f'/root/capsule/scratch/opto_tagging/metrics_{session_date}.csv'
print(f"OptoTag ready. Metrics CSV: {metrics_csv_path}")

Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json
Number of units passing the default QC: 2204
OptoTag ready. Metrics CSV: /root/capsule/scratch/opto_tagging/metrics_863973_2026-08-25_14-18-29.csv


In [9]:
# =============================================================================
# 3. COMPUTE & SAVE TAGGING METRICS (calculation only, no selection)
#    Runs the SALT / latency / jitter / reliability / effect-size metrics for
#    every unit x condition and, with per_pulse=True, ALSO separately for each
#    pulse position within the train. Includes ALL probes / shanks / lasers.
#
#    Rows are labelled by `pulse_index`:
#       -1        -> pooled over ALL pulses in the train
#        0,1,2... -> that pulse position within the train only
#
#    These CALCULATION parameters define the metrics themselves; the SELECTION
#    thresholds are applied later in step 4 (so you can re-select without
#    recomputing).
# =============================================================================
import os
import pandas as pd

metrics_df = OptoTag.compute_tagging_metrics(
    baseline_window=(-0.05, 0.0),   # baseline rate / SALT baseline epochs
    latency_window=(0.0, 0.006),    # short direct-activation window (0-6 ms)
    salt_dt=0.001,                  # SALT latency-histogram resolution (s)
    per_pulse=True,                 # also compute per pulse position in the train
    align_to_event="pulse",
    remove_artefacts=True,
    removal_window=0.001,
)

# Persist the raw metrics so selection can be re-run without recomputation.
# The `session` column (= NWB session_id) lets these per-session CSVs be
# concatenated later and selected together in step 4.
os.makedirs(os.path.dirname(metrics_csv_path), exist_ok=True)
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"Session: {OptoTag.session_name}")
print(f"Computed metrics for {len(metrics_df)} unit x condition x pulse rows.")
print(f"pulse_index values present: {sorted(metrics_df['pulse_index'].unique())}")
print(f"Saved metrics to: {metrics_csv_path}")

/opt/conda/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:430: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)


Session: ecephys_863973_2026-08-25_14-18-29
Computed metrics for 218196 unit x condition x pulse rows.
pulse_index values present: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Saved metrics to: /root/capsule/scratch/opto_tagging/metrics_863973_2026-08-25_14-18-29.csv


In [10]:
# =============================================================================
# 4. SELECT ROBUSTLY TAGGED UNITS (thresholding only; no recomputation)
#    `select_tagged_units` is a STANDALONE function (it does not need an
#    OpticalTagging instance). This notebook handles a SINGLE session, so we
#    select directly on the metrics_df from step 3. Change the thresholds below
#    and re-run this cell alone to explore different cut-offs.
# =============================================================================
from ast import literal_eval
from optical_tagging import select_tagged_units

# ---- Choose the condition to plot (all probes / all shanks are included) ----
# The condition is the 6-tuple:
#   (target_power, location_tag, laser_name, cycle_duration, frequency, pulse_duration)
#     5.0       -> target_power     : commanded laser power (mW)
#     1.0       -> location_tag      : stimulation location / site id
#     "Laser_1" -> laser_name        : which laser was used (IGNORED for selection)
#     1.0       -> cycle_duration    : laser-on duration of each train/cycle (s)
#     10.0      -> frequency         : pulse rate within the train (Hz) -> 10 pulses/s
#     0.05      -> pulse_duration    : duration of a single pulse (s) -> 50 ms
target_cond = (5.0, 1.0, "Laser_1", 1.0, 10.0, 0.05)

# ---- Choose which pulse position to select on -------------------------------
#   -1 -> pooled over ALL pulses in the train; 0 -> first pulse, 1 -> ...
SELECT_PULSE_INDEX = -1

# ---- Robust opto-tag SELECTION criteria (tune as needed) --------------------
#   effect_ratio   : require stim rate >= this x baseline rate
#   min_reliability: fraction of pulses that must evoke a spike in the window
#   max_latency    : max median first-spike latency (s)
#   max_jitter     : max SD of first-spike latency (s)
#   alpha          : applied to BH-FDR-corrected t-test AND SALT p-values
#                    (correction is done within each pulse_index family)
tagged_df = select_tagged_units(
    metrics_df,
    alpha=0.05,
    effect_ratio=2.0,
    min_abs_increase=0.0,
    min_reliability=0.2,
    max_latency=0.006,
    max_jitter=0.003,
    correction="fdr_bh",
)

# `condition` may be a string after a CSV round-trip; normalise for matching.
def as_tuple(c):
    if isinstance(c, str):
        try:
            c = literal_eval(c)
        except (ValueError, SyntaxError):
            return None
    return tuple(c) if isinstance(c, (tuple, list)) else None

# Match the condition on every field EXCEPT laser_name (index 2), so units tagged
# by any laser under the same power/location/timing are all included.
IGNORE_IDX = 2  # laser_name position in the 6-tuple
match_idx = [i for i in range(6) if i != IGNORE_IDX]

def matches_ignore_laser(c):
    c = as_tuple(c)
    if c is None or len(c) != 6:
        return False
    return all(c[i] == target_cond[i] for i in match_idx)

cond_mask = tagged_df["condition"].apply(matches_ignore_laser)
pulse_mask = tagged_df["pulse_index"] == SELECT_PULSE_INDEX

# Robustly tagged units for this condition / pulse (any probe / shank / laser).
sigrows = tagged_df[cond_mask & pulse_mask & (tagged_df["tagged"] == True)]

# Intersect with QC-passed units on the OptoTag object.
common_units = sorted(set(OptoTag.units_passing_qc.index) & set(sigrows["unit_id"]))
print(f"Condition (laser_name ignored): {target_cond} | pulse_index={SELECT_PULSE_INDEX}")
print(f"Robustly tagged units to plot ({len(common_units)}): {common_units}")

# Handy diagnostic table for the selected units.
cols = ["unit_id", "probe", "shank", "pulse_index", "reliability",
        "median_latency", "jitter", "baseline_FR_mean", "stim_FR_mean",
        "effect_ratio", "p_ttest_corr", "p_salt_corr"]
display(sigrows.sort_values("p_salt_corr")[cols].reset_index(drop=True))

Condition (laser_name ignored): (5.0, 1.0, 'Laser_1', 1.0, 10.0, 0.05) | pulse_index=-1
Robustly tagged units to plot (122): [310, 311, 748, 752, 759, 788, 792, 2023, 2025, 2036, 2059, 2109, 2115, 2116, 2149, 2185, 2200, 2203, 2208, 2209, 2212, 2215, 2225, 2230, 2233, 2238, 2239, 2240, 2241, 2245, 2250, 2255, 2257, 2259, 2263, 2264, 2272, 2277, 2284, 2288, 2289, 2296, 2300, 2303, 2314, 2315, 2317, 2322, 2324, 2326, 2331, 2334, 2344, 2366, 2558, 2586, 2587, 2588, 2609, 2613, 2629, 2630, 2634, 2685, 2689, 2692, 2706, 2732, 2735, 2803, 2811, 2826, 2845, 2946, 2947, 2960, 2983, 3000, 3014, 3020, 3023, 3031, 3032, 3033, 3258, 3269, 3326, 3329, 3335, 3336, 3385, 3393, 3400, 3403, 3453, 3467, 3502, 3603, 3620, 3629, 3656, 3663, 3665, 3673, 3696, 3700, 3705, 3718, 3736, 3757, 3809, 3814, 4045, 4062, 4077, 4106, 4159, 4227, 4270, 4273, 4280, 4320]


,unit_id,probe,shank,pulse_index,reliability,median_latency,jitter,baseline_FR_mean,stim_FR_mean,effect_ratio,p_ttest_corr,p_salt_corr
0,310,ProbeA-1,0,-1,0.837,0.003172,0.000980,61.48,193.333333,3.144654,2.398750e-176,0.0
1,311,ProbeA-1,0,-1,0.549,0.003745,0.000780,31.76,107.333333,3.379513,3.182218e-84,0.0
2,748,ProbeA-2,1,-1,0.201,0.004788,0.001214,9.32,36.166667,3.880544,1.055702e-25,0.0
3,752,ProbeA-2,1,-1,0.333,0.003578,0.001026,23.98,60.833333,2.536836,3.543393e-32,0.0
4,759,ProbeA-2,1,-1,0.738,0.004661,0.001102,61.24,130.000000,2.122796,5.852400e-104,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
117,4227,ProbeB-4,3,-1,0.303,0.005043,0.000948,7.86,52.000000,6.615776,2.257117e-57,0.0
118,4270,ProbeB-4,3,-1,0.242,0.005463,0.000572,0.16,40.333333,252.083333,4.582536e-60,0.0
119,4273,ProbeB-4,3,-1,0.322,0.004235,0.000637,7.96,55.500000,6.972362,7.581639e-62,0.0
120,4280,ProbeB-4,3,-1,0.323,0.004243,0.000684,6.12,56.833333,9.286492,5.494416e-65,0.0


In [11]:
# =============================================================================
# 5. SAVE ONE FIGURE PER TAGGED UNIT — PULSE-ALIGNED (all conditions) — PARALLEL
# =============================================================================
# plot_raster_graph draws EVERY condition for a unit in a single figure
# (one column per condition), aligned to each PULSE. Passing metrics_df also
# annotates each condition panel with the pooled (pulse_index=-1) reliability,
# median_latency, jitter and effect_ratio for this unit. Rendering is the slow
# part (lazy NWB spike reads + matplotlib), so we fan units out over processes.
import os
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

from opto_tagging_figure_worker import init_worker, render_unit

# Output folder for the pulse-aligned per-unit figures.
pulse_fig_dir = f"/root/capsule/scratch/tagged_units/{session_date}/pulse_aligned/"
os.makedirs(pulse_fig_dir, exist_ok=True)

# By default we only save PNG. Add "eps"/"pdf"/"svg" here if you also need vectors.
SAVE_FORMATS = ["png"]

# Arguments forwarded to OptoTag.plot_raster_graph for every unit.
PULSE_PLOT_KWARGS = dict(
    bin_size=0.002,
    time_window=[-0.02, 0.09],
    removal_window=0.002,
    align_to_event="pulse",
    remove_artefacts=True,
    show_waveform=True,       # mean waveform in the top-right corner
    metrics_df=metrics_df,    # annotate reliability / latency / jitter / ratio
)

MAX_WORKERS = 4  # tune to the number of cores available in the capsule

ctx = mp.get_context("spawn")  # spawn avoids NWB/HDF5 fork-safety issues
init_args = (behavior_json_file, ephys_nwb_file, pulse_fig_dir, PULSE_PLOT_KWARGS, SAVE_FORMATS)

print(f"Rendering {len(common_units)} PULSE-aligned figures to {pulse_fig_dir}")
print(f"Using {MAX_WORKERS} worker processes | formats: {SAVE_FORMATS}")
with ProcessPoolExecutor(
    max_workers=MAX_WORKERS,
    mp_context=ctx,
    initializer=init_worker,
    initargs=init_args,
) as ex:
    for unit_id, ok, msg in ex.map(render_unit, common_units):
        print(f"  unit {unit_id}: {'saved' if ok else 'FAILED: ' + msg}")

print("Done saving pulse-aligned figures.")

Rendering 122 PULSE-aligned figures to /root/capsule/scratch/tagged_units/863973_2026-08-25_14-18-29/pulse_aligned/
Using 4 worker processes | formats: ['png']
Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json
Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json
Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json
Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json
Number of units passing the default QC: 2204
Number of units passing the default QC: 2204
Number of units passing the default QC: 2204
Number of units passing the default QC: 2204
  unit 310: saved
  unit 311: saved
  unit 748: saved
  unit 752: saved
  unit 759: saved
  unit 788: saved
  unit 792: 

In [12]:
# =============================================================================
# 6. SAVE ONE FIGURE PER TAGGED UNIT — TRAIN-ALIGNED (all conditions) — PARALLEL
# =============================================================================
# Same units as step 5, but aligned to each LASER TRAIN (align_to_event="laser")
# so every pulse in the train is shaded and the full train response is visible.
# Per-pulse metrics are pulse-based, so they are NOT annotated here.
import os
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

from opto_tagging_figure_worker import init_worker, render_unit

# Output folder for the train-aligned per-unit figures.
train_fig_dir = f"/root/capsule/scratch/tagged_units/{session_date}/train_aligned/"
os.makedirs(train_fig_dir, exist_ok=True)

# Widen the window to span the whole train (cycle_duration = 1.0 s here).
TRAIN_PLOT_KWARGS = dict(
    bin_size=0.005,
    time_window=[-0.1, 1.2],
    removal_window=0.002,
    align_to_event="laser",   # align to each train instead of each pulse
    remove_artefacts=True,
    show_waveform=True,
)

ctx = mp.get_context("spawn")  # spawn avoids NWB/HDF5 fork-safety issues
init_args = (behavior_json_file, ephys_nwb_file, train_fig_dir, TRAIN_PLOT_KWARGS, SAVE_FORMATS)

print(f"Rendering {len(common_units)} TRAIN-aligned figures to {train_fig_dir}")
print(f"Using {MAX_WORKERS} worker processes | formats: {SAVE_FORMATS}")
with ProcessPoolExecutor(
    max_workers=MAX_WORKERS,
    mp_context=ctx,
    initializer=init_worker,
    initargs=init_args,
) as ex:
    for unit_id, ok, msg in ex.map(render_unit, common_units):
        print(f"  unit {unit_id}: {'saved' if ok else 'FAILED: ' + msg}")

print("Done saving train-aligned figures.")

Rendering 122 TRAIN-aligned figures to /root/capsule/scratch/tagged_units/863973_2026-08-25_14-18-29/train_aligned/
Using 4 worker processes | formats: ['png']


Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json
Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.jsonBehavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json

Behavior data successfully loaded from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json
Number of units passing the default QC: 2204
Number of units passing the default QC: 2204
Number of units passing the default QC: 2204
Number of units passing the default QC: 2204
  unit 310: saved
  unit 311: saved
  unit 748: saved
  unit 752: saved
  unit 759: saved
  unit 788: saved
  unit 792: saved
  unit 2023: saved
  unit 2025: saved
  unit 2036: saved
  unit 2059: saved
  unit 2109: saved
  unit 2115: saved
  unit 2116: saved
  unit 2149: saved
  

In [ ]:
# =============================================================================
# 7. CHECK VALUE LATENTS FOR A MODEL (QLearning_L1F1_CK1_softmax)
#    Fetch each value latent time series and verify its length matches the
#    number of valid (response) trials before using it for raster/PSTH plots.
# =============================================================================
import numpy as np
from behavior_utils import (
    get_fitted_model_names,
    extract_fitted_data,
    find_trials,
)
from nwb_utils import NWBUtils

MODEL = "QLearning_L1F1_CK1_softmax"
# Value latents to check (name). "-1" = value before the outcome.
VALUE_LATENTS = ["sumQ-1", "deltaQ-1", "chosenQ-1"]

print(f"Fitted models: {get_fitted_model_names(session_name=session_date)}")

# Load the behavior NWB (trials table) for this session.
nwb, status = NWBUtils.combine_nwb(session_name=session_date)
print(f"combine_nwb status: {status}")

# Total number of behavior trials in the NWB trials table.
n_behavior_trials = int(len(nwb.trials))
print(f"n_behavior_trials (all): {n_behavior_trials}")

# Number of valid (non-no-response) trials the latents should align to.
trials = np.asarray(find_trials(nwb, "response"))
n_trials = int(len(trials))
print(f"n_trials (response): {n_trials}\n")

# Fetch each latent and check its length.
for name in VALUE_LATENTS:
    vals = extract_fitted_data(
        nwb_behavior_data=nwb,
        session_name=session_date,
        model_alias=MODEL,
        latent_name=name,
    )
    if vals is None:
        print(f"{MODEL}-{name}: <not available>")
        continue
    vals = np.asarray(vals, dtype=float).reshape(-1)
    ok = "OK" if vals.shape[0] == n_trials else "LENGTH MISMATCH"
    print(f"{MODEL}-{name}: length={vals.shape[0]} (trials={n_trials}) -> {ok}")


['ForagingCompareThreshold',
 'WSLS',
 'QLearning_L2F1_CKfull_softmax',
 'QLearning_L1F1_CK1_softmax',
 'QLearning_L1F0_epsi',
 'QLearning_L2F1_softmax']

In [ ]:
# =============================================================================
# 8. INSPECT RAW FITTED LATENTS AT NO-RESPONSE TRIALS
#    The latents come out length=541 (all trials) but only 420 are response
#    trials -> the fit assigns values to no-response trials too. Print the raw
#    q_value at those trials to see how they are filled (NaN? carried-forward?).
# =============================================================================
import numpy as np
from behavior_utils import get_fitted_latent

fit = get_fitted_latent(session_date, MODEL)
FL = fit["fitted_latent_variables"]
print("fitted_latent_variables keys:", list(FL.keys()))

q0 = np.asarray(FL["q_value"][0], dtype=float)
q1 = np.asarray(FL["q_value"][1], dtype=float)
print(f"q_value shapes: q0={q0.shape}, q1={q1.shape}")

# Global stats: is the whole array ~0, or only certain trials?
print(f"q0 stats -> min={np.nanmin(q0):.4f}, max={np.nanmax(q0):.4f}, "
      f"mean={np.nanmean(q0):.4f}, nonzero={(np.abs(q0) > 1e-9).sum()}/{q0.size}")
print(f"q1 stats -> min={np.nanmin(q1):.4f}, max={np.nanmax(q1):.4f}, "
      f"mean={np.nanmean(q1):.4f}, nonzero={(np.abs(q1) > 1e-9).sum()}/{q1.size}")

# animal_response: 0=left, 1=right, 2=no-response.
responses = np.asarray(nwb.trials["animal_response"][:])
print(f"\nanimal_response length: {responses.shape[0]}")
print(f"counts -> left(0): {(responses==0).sum()}, "
      f"right(1): {(responses==1).sum()}, no-response(2): {(responses==2).sum()}")

# Align the q arrays to the response vector (q may carry one extra trailing state).
n = min(q0.shape[0], q1.shape[0], responses.shape[0])
q0a, q1a, resp = q0[:n], q1[:n], responses[:n]

resp_idx = np.where(resp != 2)[0]
no_resp_idx = np.where(resp == 2)[0]

# Compare the two populations: are no-response trials distinct from response ones?
sumQ = q0a + q1a
print(f"\nsumQ on RESPONSE trials     -> mean={sumQ[resp_idx].mean():.4f}, "
      f"nonzero={(np.abs(sumQ[resp_idx]) > 1e-9).sum()}/{resp_idx.size}")
print(f"sumQ on NO-RESPONSE trials  -> mean={sumQ[no_resp_idx].mean():.4f}, "
      f"nonzero={(np.abs(sumQ[no_resp_idx]) > 1e-9).sum()}/{no_resp_idx.size}")
print(f"NaN in q0 at no-response trials: {np.isnan(q0a[no_resp_idx]).sum()} / {no_resp_idx.size}")
print(f"NaN in q1 at no-response trials: {np.isnan(q1a[no_resp_idx]).sum()} / {no_resp_idx.size}")

# Show the first few no-response trials with their neighbouring values so we can
# tell whether the value was carried forward from the previous trial.
print("\nfirst no-response trials (idx | prev q0,q1 | this q0,q1 | next q0,q1):")
for i in no_resp_idx[:10]:
    prev_ = (q0a[i-1], q1a[i-1]) if i > 0 else (np.nan, np.nan)
    nxt_ = (q0a[i+1], q1a[i+1]) if i + 1 < n else (np.nan, np.nan)
    print(f"  {i:4d} | prev=({prev_[0]:.4f},{prev_[1]:.4f}) | "
          f"this=({q0a[i]:.4f},{q1a[i]:.4f}) | next=({nxt_[0]:.4f},{nxt_[1]:.4f})")

# And a few RESPONSE trials for contrast.
print("\nfirst response trials (idx | q0,q1 | sumQ):")
for i in resp_idx[:10]:
    print(f"  {i:4d} | ({q0a[i]:.4f},{q1a[i]:.4f}) | sumQ={sumQ[i]:.4f}")


In [ ]:
# =============================================================================
# 9. DUMP THE FULL fitted_latent_variables STRUCTURE
#    q_value came back all-zeros, so the real signal must live under another
#    key (or a nested layout). Print type / shape / stats for every entry, plus
#    the fitted params, to locate where the usable values actually are.
# =============================================================================
import numpy as np

def _describe(name, v, indent=0):
    pad = "  " * indent
    if isinstance(v, dict):
        print(f"{pad}{name}: dict keys={list(v.keys())}")
        for k, vv in v.items():
            _describe(k, vv, indent + 1)
        return
    if isinstance(v, (list, tuple)):
        print(f"{pad}{name}: {type(v).__name__} len={len(v)}")
        for i, vv in enumerate(v[:4]):
            _describe(f"[{i}]", vv, indent + 1)
        return
    arr = np.asarray(v, dtype=float) if not np.isscalar(v) else None
    if arr is not None and arr.size:
        nz = int((np.abs(arr) > 1e-9).sum())
        print(f"{pad}{name}: array shape={arr.shape} "
              f"min={np.nanmin(arr):.4f} max={np.nanmax(arr):.4f} "
              f"mean={np.nanmean(arr):.4f} nonzero={nz}/{arr.size}")
    else:
        print(f"{pad}{name}: {type(v).__name__} = {v}")

print("=== fitted_latent_variables ===")
for k, v in FL.items():
    _describe(k, v)

print("\n=== params ===")
params = fit.get("params")
_describe("params", params)

# Peek at the first non-zero key's leading values for a sanity check.
print("\n=== sample values (first 10) ===")
for k in ("choice_prob", "rpe", "choice_kernel"):
    if k in FL:
        arr = np.asarray(FL[k], dtype=float)
        flat = arr.reshape(arr.shape[0], -1) if arr.ndim > 1 else arr
        print(f"{k} (shape {arr.shape}):")
        print(np.array(FL[k])[..., :10] if arr.ndim > 1 else flat[:10])
